In [1]:
import warnings
# turn off warnings
warnings.filterwarnings('ignore')

# data analysis
import datetime
import numpy as np
import pandas as pd

# preprocessing
from utils import DataCleanPipleLine
from sklearn.preprocessing import StandardScaler

# viz
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from utils import plotting as plot

# OS
import os
from dotenv import load_dotenv

load_dotenv()

# set theme for the book
sns.set_theme(style="whitegrid")
plt.rcParams.update({"font.family": "sans-serif", "font.size": 11})

title_params = {
    'fontsize' : 20,
    'fontweight' : 'bold',
    'color':'grey'
}

In [2]:
# Import the data
file_location = os.getenv("F_LOC")

# run data load
cleaning_pipeline = DataCleanPipleLine.DataImportAndCleaningPipeline(file_location)

# run pipeline
window_df = cleaning_pipeline.RunPipeline()


****************************************************************************************************
DF Shape: (252803, 4)
****************************************************************************************************
****************************************************************************************************
                   DateTime  Temperature (oC)    Pressure   Humidity
252802  2026-11-13 17:15:46             15.48  1001.79160  56.400390
252803  2026-11-13 17:16:46             15.57  1001.78320  55.677736
252804  2026-11-13 17:17:46             15.62  1001.77896  55.571290
252805  2026-11-13 17:18:46             15.65  1001.74440  55.363280
252806  2026-11-13 17:19:46             15.73  1001.80070  55.060548
252807  2026-11-13 17:20:46             15.82  1001.69700  54.479492
252808  2026-11-13 17:21:47             15.79  1001.75830  54.913090
252809  2026-11-13 17:22:47             15.80  1001.72768  54.367188
252810  2026-11-13 17:23:47             15.85  1001.72

In [12]:
# import data
file_location2 = os.getenv("F_LOC2")

# run data load
aq_cleaning_pipeline = DataCleanPipleLine.DataImportAndCleaningPipeline(file_location2)

# run pipeline
door_df = aq_cleaning_pipeline.RunPipeline()

****************************************************************************************************
DF Shape: (243666, 6)
****************************************************************************************************
****************************************************************************************************
                   DateTime  Temperature (oC)    Pressure   Humidity  eCo2  \
243665  2026-11-13 17:47:33             19.21  1003.50440  46.099610   548   
243666  2026-11-13 17:48:33             19.23  1003.51152  46.045900   526   
243667  2026-11-13 17:49:33             19.18  1003.55970  46.086915   521   
243668  2026-11-13 17:50:34             19.17  1003.56090  46.073244   524   
243669  2026-11-13 17:51:34             19.17  1003.56940  46.141600   536   
243670  2026-11-13 17:52:34             19.13  1003.55496  46.137696   554   
243671  2026-11-13 17:53:34             19.13  1003.55496  46.137696   548   
243672  2026-11-13 17:54:34             19.08  1003

In [8]:
# how much missing data do we have?
min_dt = window_df.index.min()
max_dt = window_df.index.max()

# create full date range df
dt_range = pd.date_range(
    start=min_dt,
    end=max_dt,
    freq='1min'
)

dt_range = pd.DataFrame(
    dt_range,
    columns=['date']
)
dt_range.head(5)

,date
0,2026-03-29 20:54:00+00:00
1,2026-03-29 20:55:00+00:00
2,2026-03-29 20:56:00+00:00
3,2026-03-29 20:57:00+00:00
4,2026-03-29 20:58:00+00:00


In [42]:
# merge the dfs together to show missing data
window_full_dt = pd.merge(
    left=dt_range,
    right=window_df,
    left_on='date',
    right_on=window_df.index,
    how='left'
)

window_full_dt.isna().sum()/len(window_df)

date                   0.000000
Temperature (oC)       0.303669
Pressure               0.303669
Humidity               0.303669
DateTime               0.303669
Date                   0.303669
UkDate                 0.303669
YearMonth              0.303669
EndOfWeek              0.303669
Time                   0.303669
Time_Numeric           0.303669
DailyAvgTemperature    0.303669
DailyAvgHumidity       0.303669
dtype: float64

In [14]:
# merge the dfs together to show missing data
pd.merge(
    left=dt_range,
    right=door_df,
    left_on='date',
    right_on=door_df.index,
    how='left'
).isna().sum()/len(window_df)

date                   0.000000
Temperature (oC)       0.339961
Pressure               0.339961
Humidity               0.339961
eCo2                   0.339961
VOC                    0.339961
DateTime               0.339961
Date                   0.339961
UkDate                 0.339961
YearMonth              0.339961
EndOfWeek              0.339961
Time                   0.339961
Time_Numeric           0.339961
DailyAvgTemperature    0.339961
DailyAvgHumidity       0.339961
dtype: float64

##### Notes

We have a considerable amount of data missing from both dataframe (30% and 33%). The next step is to determine which method is a good approach to impute the data.

The options to explore are:

1. The mean.
2. Interpolation.

In [47]:
# explore the mean and it's impact on the data

missing_data_door_df_stats = door_df.describe()[['Humidity','Temperature (oC)']]
missing_data_window_df_stats= window_df.describe()[['Humidity','Temperature (oC)']]

# take the mean values
door_df_mean_hum, door_df_mean_tmp  = missing_data_door_df_stats.iloc[1].values
window_df_mean_hum, window_df_mean_tmp  = missing_data_window_df_stats.iloc[1].values

# fill in missing data with the mean
window_full_dt['Humidity'] = window_full_dt['Humidity'].fillna(window_df_mean_hum)
window_full_dt['Temperature (oC)'] = window_full_dt['Temperature (oC)'].fillna(window_df_mean_tmp)

# show how the data distributions have changed
print("Data with NaN")
print(missing_data_window_df_stats)
print(f"\n")
print("Data with NaN imputed")
print(window_full_dt[['Humidity','Temperature (oC)']].describe())

Data with NaN
            Humidity  Temperature (oC)
count  252788.000000     252788.000000
mean       52.747710         19.631088
min        26.669922         13.080000
25%        46.401368         17.180000
50%        53.474610         19.320000
75%        59.527344         21.690000
max        77.616210         29.740000
std         8.544590          3.018795


Data with NaN imputed
            Humidity  Temperature (oC)
count  329551.000000     329551.000000
mean       52.747695         19.631099
std         7.483550          2.643925
min        26.669922         13.080000
25%        48.780272         17.810000
50%        52.747710         19.631088
75%        57.670900         21.010000
max        77.616210         29.740000


##### Notes

It appears we have shrunk the distributions for both samples, as we have forced the data closer to the sample mean. The problem with this method is that we have artifically forced the distribution to be thinner (smaller standard deviation) and pull closer to the sample mean. The consequences are underestimated variance (overconfident intervals and tests) and weakened relationships between this feature and the others, since the imputed values carry no information about them.

If the number of missing values was smaller (<5% for example), I wouldn't be too concerned, but because we are dealing with a large amount of missing data, we are effectively altering our sample distribution in a negative way.

In [39]:
missing_data_door_df_stats.iloc[1].values

array([47.29548925, 22.16819295])

In [40]:
door_df_mean_humidity

np.float64(22.168192952069806)